# Hallucination Detection & Faithfulness Scoring in LLM Outputs
## UE23AM343BA2 — LLM Applications Jackfruit Project

### Pipeline Overview
- **NLI Scorer**: DeBERTa-v3 for sentence-level entailment detection
- **Semantic Scorer**: Sentence-Transformers for BERTScore-style similarity
- **LLM Judge**: Gemini 1.5 Flash for reasoning-level hallucination detection
- **HalluScore**: Novel composite metric combining all three signals

---

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
print('Imports successful!')

## 1. Load the Benchmark Dataset

In [ ]:
from benchmark import BenchmarkLoader

loader = BenchmarkLoader()
stats  = loader.get_statistics()

print('=== Benchmark Statistics ===')
print(f'Total Samples:        {stats["total_samples"]}')
print(f'Faithful:             {stats["faithful_samples"]}')
print(f'Hallucinated:         {stats["hallucinated_samples"]}')
print(f'Balance Ratio:        {stats["balance_ratio"]:.2f}')
print(f'Domains:              {list(stats["domains"].keys())}')
print(f'Hallucination Types:  {list(stats["hallucination_types"].keys())}')

## 2. Run NLI Scoring (DeBERTa-v3)

In [ ]:
from nli_scorer import NLIScorer

nli = NLIScorer()

# Test on a faithful example
context   = "Photosynthesis occurs in chloroplasts and uses sunlight, water, and CO2 to produce glucose."
faithful  = "Plants use sunlight and CO2 to produce glucose in chloroplasts."
hallucin  = "Photosynthesis occurs in mitochondria using nitrogen and UV light to produce carbon dioxide."

r1 = nli.score_claim(context, faithful)
r2 = nli.score_claim(context, hallucin)

print('\n--- Faithful Claim ---')
print(f'  Entailment:     {r1["entailment_prob"]:.4f}')
print(f'  Neutral:        {r1["neutral_prob"]:.4f}')
print(f'  Contradiction:  {r1["contradiction_prob"]:.4f}')
print(f'  Predicted:      {r1["predicted_label"]}')
print(f'  Faithfulness:   {r1["faithfulness_score"]:.4f}')

print('\n--- Hallucinated Claim ---')
print(f'  Entailment:     {r2["entailment_prob"]:.4f}')
print(f'  Neutral:        {r2["neutral_prob"]:.4f}')
print(f'  Contradiction:  {r2["contradiction_prob"]:.4f}')
print(f'  Predicted:      {r2["predicted_label"]}')
print(f'  Faithfulness:   {r2["faithfulness_score"]:.4f}')

## 3. Run Semantic Scoring

In [ ]:
from semantic_scorer import SemanticScorer

sem = SemanticScorer()

s1 = sem.score_claim(context, faithful)
s2 = sem.score_claim(context, hallucin)

print('\n--- Semantic Similarity ---')
print(f'  Faithful   cosine sim: {s1["cosine_similarity"]:.4f} | score: {s1["faithfulness_score"]:.4f}')
print(f'  Hallucin.  cosine sim: {s2["cosine_similarity"]:.4f} | score: {s2["faithfulness_score"]:.4f}')

## 4. Run Gemini LLM Judge

In [ ]:
from llm_judge import GeminiJudge

# Set your Gemini API key
API_KEY = 'YOUR_GEMINI_API_KEY_HERE'

judge = GeminiJudge(api_key=API_KEY)

j1 = judge.score_claim(context, faithful)
j2 = judge.score_claim(context, hallucin)

print('\n--- Gemini Judge ---')
print(f'  Faithful:    verdict={j1["verdict"]}  score={j1["faithfulness_score"]:.4f}  reason={j1["reason"]}')
print(f'  Hallucin.:   verdict={j2["verdict"]}  score={j2["faithfulness_score"]:.4f}  reason={j2["reason"]}')

## 5. Compute HalluScore (Composite)

In [ ]:
from hallu_score import HalluScoreCalculator

calc = HalluScoreCalculator()

# Score full documents
nli_doc1 = nli.score_document(context, faithful)
sem_doc1 = sem.score_document(context, faithful)
llm_doc1 = judge.score_document(context, faithful)
hs1 = calc.compute(nli_doc1, sem_doc1, llm_doc1, model_name='faithful_example')

nli_doc2 = nli.score_document(context, hallucin)
sem_doc2 = sem.score_document(context, hallucin)
llm_doc2 = judge.score_document(context, hallucin)
hs2 = calc.compute(nli_doc2, sem_doc2, llm_doc2, model_name='hallucinated_example')

print('\n========== HalluScore Results ==========')
print(f'Faithful Example:')
print(f'  HalluScore: {hs1.hallu_score:.4f}  Verdict: {hs1.verdict}  Confidence: {hs1.confidence:.4f}')
print(f'  NLI={hs1.nli_score:.3f}  Semantic={hs1.semantic_score:.3f}  LLM={hs1.llm_judge_score:.3f}')
print(f'  95% CI: [{hs1.confidence_interval[0]:.3f}, {hs1.confidence_interval[1]:.3f}]')

print(f'\nHallucinated Example:')
print(f'  HalluScore: {hs2.hallu_score:.4f}  Verdict: {hs2.verdict}  Confidence: {hs2.confidence:.4f}')
print(f'  NLI={hs2.nli_score:.3f}  Semantic={hs2.semantic_score:.3f}  LLM={hs2.llm_judge_score:.3f}')

## 6. Run Full Benchmark Evaluation

In [ ]:
# This runs the full pipeline on all benchmark samples
# Replace with your API key

from evaluator import HallucinationEvaluator

evaluator = HallucinationEvaluator(api_key=API_KEY)
results   = evaluator.run_benchmark(max_samples=None)  # set to small number to test
evaluator.save_results(results)

metrics = results['evaluation_metrics']
print('\n=== Final Metrics ===')
for k, v in metrics.items():
    print(f'  {k}: {v}')

## 7. Visualizations

In [ ]:
# Load results and generate all plots
from analysis import ResultsAnalyzer

analyzer = ResultsAnalyzer('../results/benchmark_results.json')
analyzer.generate_all_plots()
print('All plots saved to results/')

In [ ]:
# Display key plot inline
from IPython.display import Image
Image('../results/score_distributions.png')

In [ ]:
Image('../results/domain_accuracy.png')

## 8. HalluScore vs Baselines: Ablation Study

In [ ]:
# Compare contribution of each component
import json
import numpy as np

with open('../results/benchmark_results.json') as f:
    res = json.load(f)

samples = res['per_sample_results']

# Build comparison table
rows = []
for r in samples:
    if 'summary' not in r or 'ground_truth' not in r:
        continue
    s  = r['summary']
    gt = r['ground_truth']['is_faithful']
    rows.append({
        'id':           r['id'],
        'hallu_score':  s['hallu_score'],
        'nli_score':    s['nli_score'],
        'semantic':     s['semantic_score'],
        'llm_judge':    s['llm_judge_score'],
        'is_faithful':  int(gt)
    })

df = pd.DataFrame(rows)

print('Score Separation (mean per label):')
print(df.groupby('is_faithful')[['hallu_score','nli_score','semantic','llm_judge']].mean().round(4))

# Correlation with ground truth
from scipy.stats import spearmanr
print('\nSpearman Correlation with Ground Truth:')
for col in ['hallu_score', 'nli_score', 'semantic', 'llm_judge']:
    r, p = spearmanr(df[col], df['is_faithful'])
    print(f'  {col:20s}: r={r:.4f}, p={p:.4f}')

---
## Summary
This notebook demonstrated the complete **HalluDetect** pipeline:
1. **NLI-based detection** using DeBERTa-v3 sentence entailment
2. **Semantic similarity** using BERTScore-style precision/recall/F1
3. **LLM-as-judge** using Gemini 1.5 Flash for nuanced reasoning
4. **HalluScore**: a novel composite metric outperforming individual baselines

Results show that multi-signal combination consistently outperforms any single metric alone.